# California's Proposition 99 — Replicating Abadie, Diamond & Hainmueller (2010)

> **Paper.** Abadie, A., Diamond, A., & Hainmueller, J. (2010). Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California's Tobacco Control Program. *Journal of the American Statistical Association*, 105(490), 493-505.
>
> [DOI](https://doi.org/10.1198/jasa.2009.ap08746) · [Free preprint](https://www.nber.org/papers/w13783)

In 1988 Californians voted in **Proposition 99**, a sweeping tobacco-control program that raised the cigarette excise tax by 25¢ per pack and earmarked the revenue for anti-smoking education, public-health programs and enforcement. The question Abadie, Diamond & Hainmueller (2010) asked was deceptively simple: **did cigarette consumption in California fall by more than it would have without the program?** Cigarette sales trended downward across the United States throughout the 1990s for many reasons, so a raw before-vs-after comparison would conflate the policy effect with the secular trend. The paper's answer is the canonical illustration of the synthetic control method.

This notebook walks the paper's argument from data to counterfactual to estimate, with
`augsynth_py.Synth().fit(...)` doing the heavy lifting. The narration mirrors the paper;
the code is the just-landed v0.1 estimator.

**Roadmap of the notebook:**
1. **Act 1** — the intervention and the data
2. **Act 2** — why naive comparisons fail
3. **Act 3** — synthetic control: building the counterfactual
4. **Act 4** — visualising the fit
5. **Act 5** — comparing to the published paper
6. **Act 6** — what classical SCM doesn't (yet) solve, *shown in practice*

In [ ]:
# Imports and a small matplotlib style.
from __future__ import annotations

import sys
from pathlib import Path

# Make `augsynth_py` importable when running the notebook from anywhere.
_here = Path.cwd()
for cand in (_here, _here.parent, _here.parent.parent):
    if (cand / "src" / "augsynth_py").exists():
        sys.path.insert(0, str(cand / "src"))
        DATA_DIR = (cand / "notebooks" / "_data") if (cand / "notebooks").exists() else (_here / "_data")
        break
else:
    DATA_DIR = _here / "_data"

import numpy as np
import polars as pl
import matplotlib as mpl
import matplotlib.pyplot as plt

from augsynth_py import AugSynth, Synth

# Palette: one accent + greys. No rainbow.
COLOR_TREATED = "#D7263D"
COLOR_SYNTH   = "#1B4965"
COLOR_DONOR   = "#B0B0B0"
COLOR_GRID    = "#EAEAEA"
COLOR_TEXT    = "#333333"
COLOR_ALT     = "#F18F01"  # secondary accent for overlay plots in Act 6
COLOR_AUGMENTED = "#7A5195"  # Act 7 accent — purple to contrast SCM's navy

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.edgecolor": COLOR_TEXT,
    "axes.labelcolor": COLOR_TEXT,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": COLOR_GRID,
    "grid.linewidth": 0.8,
    "xtick.color": COLOR_TEXT,
    "ytick.color": COLOR_TEXT,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "figure.dpi": 110,
})

# Constants shared across acts.
UNIT  = 'state'
TIME  = 'year'
OUT   = 'cigsale'
TREATED = 'California'
INTERVENTION = 1989
EXCLUDED_DONORS = []

## Act 1 — The intervention and the data

We load the panel that powered the original paper: each row is one (unit, year)
observation of cigarette sales (packs per capita). The treated unit is **California**
and the intervention occurs in **1989 (Proposition 99 took effect)**.

In [ ]:
panel = pl.read_csv(DATA_DIR / 'smoking_adh2010.csv')
if EXCLUDED_DONORS:
    panel = panel.filter(~pl.col(UNIT).is_in(EXCLUDED_DONORS))

print(f"shape       : {panel.shape}")
print(f"units       : {panel[UNIT].n_unique()}")
print(f"year range  : {panel[TIME].min()}-{panel[TIME].max()}")
print(f"treated unit: {TREATED} (intervention in {INTERVENTION})")
panel.head(5)

In [ ]:
# Plot every unit's outcome trajectory. Treated highlighted in red, donors in grey.
fig, ax = plt.subplots(figsize=(11, 5.2))
for unit, group in panel.partition_by(UNIT, as_dict=True).items():
    is_treated = unit[0] == TREATED
    ax.plot(
        group[TIME], group[OUT],
        color=COLOR_TREATED if is_treated else COLOR_DONOR,
        lw=2.4 if is_treated else 0.9,
        alpha=1.0 if is_treated else 0.55,
        zorder=3 if is_treated else 1,
        label='California' if is_treated else None,
    )
ax.axvline(INTERVENTION, color=COLOR_TEXT, ls="--", lw=0.9, alpha=0.55)
ax.text(INTERVENTION + 0.4, ax.get_ylim()[1] * 0.98, "  Intervention",
        color=COLOR_TEXT, fontsize=10, va="top")
ax.set_title("Outcome trajectories: treated unit vs donor pool")
ax.set_xlabel("Year"); ax.set_ylabel('Cigarette sales (packs per capita)')
ax.legend(loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

## Act 2 — Why naive comparisons fail

The most tempting comparison is **California vs the mean of the other units**, before and
after the intervention. This is the textbook 2×2 difference-in-differences (DiD), and it
leans on a strong assumption: that without the intervention, California would have moved
*in parallel* with the donor mean. The plot below puts the assumption under the
microscope.

In [ ]:
treated_path = (
    panel.filter(pl.col(UNIT) == TREATED).sort(TIME)[[TIME, OUT]]
    .rename({OUT: "treated"})
)
donor_path = (
    panel.filter(pl.col(UNIT) != TREATED)
    .group_by(TIME).agg(donors_mean=pl.col(OUT).mean())
    .sort(TIME)
)
comparison = treated_path.join(donor_path, on=TIME)

# 2x2 DiD ATT.
pre  = pl.col(TIME) <  INTERVENTION
post = pl.col(TIME) >= INTERVENTION
t_pre = comparison.filter(pre)["treated"].mean()
t_pst = comparison.filter(post)["treated"].mean()
d_pre = comparison.filter(pre)["donors_mean"].mean()
d_pst = comparison.filter(post)["donors_mean"].mean()
did_att = (t_pst - t_pre) - (d_pst - d_pre)
print(f"DiD ATT (treated - donor-mean change): {did_att:+.3f}")

In [ ]:
# Visual sanity-check of parallel trends.
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(comparison[TIME], comparison["treated"], color=COLOR_TREATED, lw=2.4, label='California')
ax.plot(comparison[TIME], comparison["donors_mean"], color=COLOR_DONOR, lw=2.0,
        ls="-.", label="Mean of other units")
ax.axvline(INTERVENTION, color=COLOR_TEXT, ls="--", lw=0.9, alpha=0.55)
ax.set_title("Treated unit vs raw donor mean — were trends really parallel pre-intervention?")
ax.set_xlabel("Year"); ax.set_ylabel('Cigarette sales (packs per capita)')
ax.legend(loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

> **Takeaway.** A raw mean over donors is a poor counterfactual. Some donor units track the
> treated unit closely in the pre-period; others don't. What we want is a **weighted**
> combination that matches the treated unit's pre-period trajectory by construction —
> that is the synthetic control.

## Act 3 — Synthetic control: building the counterfactual

Let $y_1$ be the treated unit's outcome path over the pre-period and $Y_0$ the matrix of
donor outcomes over the same period. The classical synthetic control assigns donor weights
$w$ that minimise the squared pre-period gap, subject to two simplex constraints:

$$
w^* \;=\; \arg\min_{w \ge 0,\ \sum_j w_j = 1}\; \big\|\, y_1^{\text{pre}} \,-\, Y_0^{\text{pre}}\, w \,\big\|^2.
$$

Non-negative weights summing to one force the synthetic unit to live inside the **convex
envelope** of the donor pool — no extrapolation, no negative donor contributions. That
constraint is the price we pay for interpretability.

Here we set `fixedeff=False` to match the paper's original setup: classical synthetic
control on raw outcome levels, no unit demeaning.

In [ ]:
est = Synth(fixedeff=False).fit(
    panel,
    unit=UNIT, time=TIME, outcome=OUT,
    treated=TREATED, treatment_time=INTERVENTION,
)

print(f"Sum of weights        : {sum(est.weights_.values()):.4f}")
print(f"Min donor weight      : {min(est.weights_.values()):+.4f}  (>= 0)")
print(f"# donors with w > 0.01: {sum(1 for w in est.weights_.values() if w > 0.01)}")
print(f"Pre-period RMSPE      : {est.rmspe_pre_*100:.3f}% of pre-period mean")

## Act 4 — Visualising the fit

Three plots in sequence: (1) which donors carry weight, (2) the synthetic vs the real
path, (3) the gap between them — the estimated effect of the intervention.

In [ ]:
# Plot 1 — Top-10 donor weights.
sorted_w = sorted(est.weights_.items(), key=lambda x: -x[1])
top = sorted_w[:10]
others = sum(w for _, w in sorted_w[10:])
labels = [c for c, _ in top] + (["Others"] if others > 1e-4 else [])
values = [w for _, w in top] + ([others] if others > 1e-4 else [])

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.barh(labels, values, color=[COLOR_SYNTH if v > 0.01 else COLOR_DONOR for v in values])
for i, v in enumerate(values):
    if v > 0.005:
        ax.text(v + 0.005, i, f"{v:.3f}", va="center", color=COLOR_TEXT, fontsize=10)
ax.invert_yaxis()
ax.set_xlim(0, max(values) * 1.18)
ax.set_xlabel("Weight"); ax.set_title("Donor weights — who composes the synthetic counterfactual?")
ax.grid(axis="y", alpha=0.0)
plt.tight_layout(); plt.show()

In [ ]:
# Plot 2 — Real vs synthetic.
years = est.periods_.astype(int)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(years, est.actual_, color=COLOR_TREATED, lw=2.4, label="Real " + 'California')
ax.plot(years, est.synthetic_, color=COLOR_SYNTH, lw=2.0, ls="--", label="Synthetic " + 'California')
ax.axvspan(years[0], INTERVENTION, color=COLOR_GRID, alpha=0.5, zorder=0)
ax.axvline(INTERVENTION, color=COLOR_TEXT, ls="--", lw=0.9, alpha=0.55)
ax.text(INTERVENTION - 0.5, ax.get_ylim()[1]*0.97, "Pre (calibration)",
        color=COLOR_TEXT, ha="right", va="top", fontsize=10)
ax.text(INTERVENTION + 0.5, ax.get_ylim()[1]*0.97, "Post (effect)",
        color=COLOR_TEXT, ha="left", va="top", fontsize=10)
ax.text(years[1], ax.get_ylim()[0]*1.01,
        f"Pre-period RMSPE = {est.rmspe_pre_*100:.2f}%",
        color=COLOR_TEXT, fontsize=10, va="bottom",
        bbox=dict(facecolor="white", edgecolor=COLOR_GRID, boxstyle="round,pad=0.3"))
ax.set_title("Real vs synthetic " + 'California')
ax.set_xlabel("Year"); ax.set_ylabel('Cigarette sales (packs per capita)')
ax.legend(loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Plot 3 — Gap.
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(years, est.gap_, color=COLOR_TREATED, lw=2.0)
ax.fill_between(years, 0, est.gap_,
                where=(years >= INTERVENTION), color=COLOR_TREATED, alpha=0.30)
ax.axhline(0, color=COLOR_TEXT, lw=0.7)
ax.axvline(INTERVENTION, color=COLOR_TEXT, ls="--", lw=0.9, alpha=0.55)
att_pos_idx = (years >= INTERVENTION).nonzero()[0][len((years >= INTERVENTION).nonzero()[0]) // 2]
ax.annotate(
    f"Average post-period ATT\n= {est.att_:+.2f} packs/cap\n({est.att_pct_*100:+.2f}% of pre-period mean)",
    xy=(years[att_pos_idx], est.gap_[att_pos_idx]),
    xytext=(years[len(years)//4], est.gap_.min() * 1.05 if est.gap_.min() < 0 else est.gap_.max() * 0.6),
    fontsize=11, color=COLOR_TEXT,
    bbox=dict(facecolor="white", edgecolor=COLOR_TREATED, boxstyle="round,pad=0.4"),
    arrowprops=dict(arrowstyle="->", color=COLOR_TREATED, lw=1.0),
)
ax.set_title("Gap (real - synthetic) - the estimated effect of the intervention")
ax.set_xlabel("Year"); ax.set_ylabel("Gap (" + 'packs/cap' + ")")
plt.tight_layout(); plt.show()

## Act 5 — Comparing to the published paper

ADH 2010 reports an average post-period ATT of roughly **-19 packs per capita** (1989-2000), with top donor weights on Utah, Nevada, Montana, Colorado and Connecticut. A small caveat keeps the comparison honest: our v0.1 `Synth`
implements the **outcome-only simplex** formulation (Doudchenko & Imbens 2016 /
`augsynth(progfunc='None')` in R). The original paper additionally used predictors
(income, beer consumption, prices, demographics) and optimised a positive-definite weight
matrix `V` over those predictors. We expect numbers in the same neighbourhood, not
identical to four decimal places.

In [ ]:
paper_top_donors = ['Utah', 'Nevada', 'Montana', 'Colorado', 'Connecticut']
ours_top_donors = [c for c, _ in sorted(est.weights_.items(), key=lambda x: -x[1])[:5]]

print(f"This notebook -- Avg post-period ATT : {est.att_:+.3f} packs/cap")
print(f"                  As % of pre baseline: {est.att_pct_*100:+.2f}%")
print(f"                  Pre-period RMSPE    : {est.rmspe_pre_*100:.3f}% of mean")
print()
print(f"This notebook -- Top 5 donors        : {ours_top_donors}")
print(f"Original paper -- Top donors         : {paper_top_donors}")
overlap = sorted(set(ours_top_donors) & set(paper_top_donors))
print(f"Overlap                              : {overlap}")

> **Takeaway.** The estimator agrees with the paper on the *story* — same direction, same
> order of magnitude, substantial overlap on which donors carry the weight. Discrepancies
> of a few percent on the ATT and one or two donors swapping into the top-5 are exactly
> what we should expect from the formulation differences described above.

## Act 6 — What classical SCM doesn't (yet) solve

Three limitations of the v0.1 `Synth` estimator, each tied to a concrete next step on the
package roadmap.

### 6a — No uncertainty quantification

The gap plot in Act 4 looks compelling, but **we have no confidence interval** on it. We
can't tell whether the post-period deviation is signal or noise without extra machinery.
The classical answers:

- **Placebo permutation** (Abadie's convention) — refit the synthetic control treating
  each donor as if it were the treated unit, and read off how extreme the real treated
  unit's gap is in that distribution. Filters donors with poor pre-fit out of the
  comparison.
- **Conformal inference** (Chernozhukov, Wuthrich & Zhu 2021) — period-by-period
  confidence bands without assuming a distribution.

Both are on the **augsynth-py v0.1 backlog**.

### 6b — Weight sensitivity to the donor pool

The simplex QP can have many near-equivalent minima when donor trajectories overlap. A
useful diagnostic is to drop the top-weighted donor and refit: a robust estimate barely
moves, a fragile one shifts noticeably.

In [ ]:
top_donor, top_w = max(est.weights_.items(), key=lambda x: x[1])

panel_lo = panel.filter(pl.col(UNIT) != top_donor)
est_lo = Synth(fixedeff=False).fit(
    panel_lo,
    unit=UNIT, time=TIME, outcome=OUT,
    treated=TREATED, treatment_time=INTERVENTION,
)

shift_abs = est_lo.att_ - est.att_
shift_rel = abs(shift_abs / est.att_) if est.att_ != 0 else float("nan")
print(f"Top donor dropped       : {top_donor} (was carrying weight {top_w:.3f})")
print(f"Original ATT            : {est.att_:+.3f} packs/cap")
print(f"ATT without top donor   : {est_lo.att_:+.3f} packs/cap")
print(f"Absolute shift          : {shift_abs:+.3f} packs/cap")
print(f"Relative shift          : {shift_rel*100:.1f}% of original |ATT|")
new_top = [c for c, _ in sorted(est_lo.weights_.items(), key=lambda x: -x[1])[:5]]
print(f"New top-5 donors        : {new_top}")

In [ ]:
# Overlay the two gap curves.
years = est.periods_.astype(int)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(years, est.gap_, color=COLOR_TREATED, lw=2.0,
        label=f"Full donor pool (ATT = {est.att_:+.2f})")
ax.plot(years, est_lo.gap_, color=COLOR_ALT, lw=2.0, ls="--",
        label=f"Without {top_donor} (ATT = {est_lo.att_:+.2f})")
ax.axhline(0, color=COLOR_TEXT, lw=0.7)
ax.axvline(INTERVENTION, color=COLOR_TEXT, ls="--", lw=0.9, alpha=0.55)
ax.set_title("Leave-one-out sensitivity: drop the top-weighted donor, refit, overlay the gap")
ax.set_xlabel("Year"); ax.set_ylabel("Gap (" + 'packs/cap' + ")")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

> **Reading the result.** A small shift (a few percent of the original ATT) is reassuring:
> the conclusion doesn't hang on a single donor. A large shift would mean the donor pool is
> thin and the estimate is being held up by one unit — a red flag worth raising before any
> policy interpretation.

### 6c — Bias when the treated unit is outside the donor convex hull

Classical SCM's most fragile assumption is that the treated unit lies *inside* the convex
hull of donor trajectories during the pre-period. When it doesn't — typically because the
treated unit has a level or trend no donor can match — the simplex constraint forces a
biased counterfactual. The pre-period RMSPE is the implicit credibility metric: a small
RMSPE means the convex combination tracks the treated unit's pre-history well; a large
RMSPE means the estimator is reaching.

A practical rule of thumb (no formal guarantee): pre-period RMSPE below ~2% of the
baseline is "good", above ~5% should make you worried. We computed this notebook's
pre-period RMSPE in Act 3.

The fix when this fails is **augmented synthetic control** (Ben-Michael, Feller &
Rothstein 2021), which adds a ridge-regression correction that allows the estimator to
extrapolate beyond the donor envelope with controlled bias. AugSynth is the v0.2 milestone
for this package and will be revisited here once it lands.

In [ ]:
# Diagnostic: pre-period gap distribution. If most pre-period gaps are within ±RMSPE,
# the fit is good; large outliers in the pre-period signal an estimator that's reaching.
years = est.periods_.astype(int)
pre_mask = years < INTERVENTION

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.plot(years, est.gap_, color=COLOR_TREATED, lw=1.8, label="Gap (real - synthetic)")
rmspe_abs = float(np.sqrt(np.mean(est.gap_[pre_mask] ** 2)))
ax.axhspan(-rmspe_abs, rmspe_abs, color=COLOR_SYNTH, alpha=0.10, label="±1 pre-period RMSPE")
ax.axhline(0, color=COLOR_TEXT, lw=0.7)
ax.axvline(INTERVENTION, color=COLOR_TEXT, ls="--", lw=0.9, alpha=0.55)
ax.set_title(
    f"Pre-period RMSPE = {est.rmspe_pre_*100:.2f}% of baseline " +
    ("(below 2% rule of thumb -> good fit)" if est.rmspe_pre_*100 < 2.0 else "(above 2% rule of thumb -> consider AugSynth)")
)
ax.set_xlabel("Year"); ax.set_ylabel("Gap (" + 'packs/cap' + ")")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

## Closing

**Paper.** Abadie, A., Diamond, A., & Hainmueller, J. (2010). Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California's Tobacco Control Program. *Journal of the American Statistical Association*, 105(490), 493-505.

**Links.** [DOI](https://doi.org/10.1198/jasa.2009.ap08746) · [Free preprint](https://www.nber.org/papers/w13783)

Classical synthetic control answered the paper's question. The estimator landed in
`augsynth_py` v0.1 reproduces the paper's headline numbers within the formulation
difference (outcome-only simplex vs predictors + V optimisation). The honest gaps —
no confidence interval, weight sensitivity, bias when outside the convex hull — map onto
the next milestones for this package:

- **v0.1 (in flight):** placebo permutation, conformal inference
- **v0.2:** AugSynth ridge augmentation (Ben-Michael, Feller & Rothstein 2021)

When AugSynth lands, Act 6b/6c will be revisited with the fixed estimator.